# Healthcare Admission Forecasting — Synthetic Demo

**Privacy notice:** This notebook uses entirely synthetic fictional admission data. It contains no real hospital records, patient information, or real-world model results.

It compares ETS and Recent Average using walk-forward forecasts for January–October 2026, calculates metrics, selects a model per channel, forecasts November–December 2026, and creates exactly two charts.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pathlib import Path
from IPython.display import display

# 1. Load the bundled synthetic data
DATA_FILE = Path("sample_data/synthetic_admission_data.xlsx")
if not DATA_FILE.exists():
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = Path(next(iter(uploaded.keys())))

daily = pd.read_excel(DATA_FILE)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values("Date").set_index("Date")
channels = ["Cash", "CGHS", "Corporate", "EWS", "International", "TPA"]
daily = daily[channels].apply(pd.to_numeric, errors="coerce").fillna(0)

# 2. Aggregate to monthly totals
monthly = daily.resample("MS").sum()

# 3. Forecast methods
def forecast_ets(series, steps=1):
    series = series.dropna().astype(float)
    if len(series) < 24:
        raise ValueError("ETS requires at least 24 monthly observations.")
    model = ExponentialSmoothing(
        series, trend="add", damped_trend=True,
        seasonal="add", seasonal_periods=12,
        initialization_method="estimated"
    ).fit()
    return np.maximum(0, model.forecast(steps).to_numpy())

def forecast_recent_average(series, steps=1):
    series = series.dropna().astype(float)
    value = max(0, float(series.tail(4).mean())) if len(series) else 0
    return np.repeat(value, steps)

def get_forecast(series, method, steps=1):
    return forecast_ets(series, steps) if method == "ETS" else forecast_recent_average(series, steps)

# 4. Walk-forward forecasts for Jan-Oct 2026
test_months = pd.date_range("2026-01-01", "2026-10-01", freq="MS")
methods = ["ETS", "Recent Average"]
records = []

for target in test_months:
    train = monthly.loc[monthly.index < target]
    actual_row = monthly.loc[target]
    for channel in channels:
        for method in methods:
            try:
                prediction = get_forecast(train[channel], method, 1)[0]
            except Exception:
                prediction = np.nan
            records.append({
                "Month": target, "Channel": channel, "Model": method,
                "Actual": float(actual_row[channel]), "Predicted": float(prediction)
            })

validation = pd.DataFrame(records).dropna(subset=["Predicted"])
validation["Error"] = validation["Predicted"] - validation["Actual"]

# 5. Calculate metrics
def calculate_metrics(group):
    actual = group["Actual"].astype(float)
    error = group["Predicted"].astype(float) - actual
    nonzero = actual > 0
    return pd.Series({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "MAPE (%)": (error[nonzero].abs() / actual[nonzero]).mean() * 100 if nonzero.any() else np.nan,
        "WAPE (%)": error.abs().sum() / actual.sum() * 100 if actual.sum() else np.nan,
        "Bias": error.mean(),
        "Bias (%)": error.sum() / actual.sum() * 100 if actual.sum() else np.nan
    })

overall_stats = validation.groupby("Model").apply(calculate_metrics, include_groups=False).reset_index()
channel_stats = validation.groupby(["Channel", "Model"]).apply(calculate_metrics, include_groups=False).reset_index()
best_channel_models = channel_stats.sort_values("WAPE (%)").drop_duplicates("Channel").reset_index(drop=True)

print("OVERALL MODEL PERFORMANCE — SYNTHETIC DATA")
display(overall_stats.round(2))
print("CHANNEL-WISE MODEL PERFORMANCE — SYNTHETIC DATA")
display(channel_stats.round(2))
print("BEST MODEL BY CHANNEL — SYNTHETIC DATA")
display(best_channel_models[["Channel", "Model", "WAPE (%)", "MAPE (%)"]].round(2))

# 6. Forecast Nov-Dec 2026 using best model per channel
future_months = pd.date_range("2026-11-01", "2026-12-01", freq="MS")
selected_method = dict(zip(best_channel_models["Channel"], best_channel_models["Model"]))
future_predictions = pd.DataFrame(index=channels)
for channel in channels:
    method = selected_method[channel]
    future_predictions.loc[channel, [m.strftime("%b %Y") for m in future_months]] = get_forecast(monthly[channel], method, 2)
future_predictions = future_predictions.astype(float)
print("NOVEMBER-DECEMBER 2026 FORECAST — SYNTHETIC DATA")
display(future_predictions.round(0))

# 7. Actual vs predicted using the selected method per channel (Jan-Oct)
selected_parts = []
for channel in channels:
    method = selected_method[channel]
    selected_parts.append(validation[(validation["Channel"] == channel) & (validation["Model"] == method)])
selected_validation = pd.concat(selected_parts, ignore_index=True)

monthly_comparison = pd.DataFrame({
    "Month": test_months.strftime("%b %Y"),
    "Actual Admissions": monthly.loc[test_months, channels].sum(axis=1).to_numpy(),
    "Predicted Admissions": selected_validation.groupby("Month")["Predicted"].sum().reindex(test_months).to_numpy()
})
channel_comparison = pd.DataFrame({
    "Channel": channels,
    "Actual Admissions": [selected_validation[selected_validation["Channel"] == ch]["Actual"].sum() for ch in channels],
    "Predicted Admissions": [selected_validation[selected_validation["Channel"] == ch]["Predicted"].sum() for ch in channels]
})
print("MONTHLY ACTUAL VS PREDICTED — SYNTHETIC DATA")
display(monthly_comparison.round(0))
print("CHANNEL-WISE ACTUAL VS PREDICTED — SYNTHETIC DATA")
display(channel_comparison.round(0))

# 8. Graph 1 — Actual vs Predicted Monthly (2026)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=monthly_comparison["Month"], y=monthly_comparison["Actual Admissions"],
    mode="lines+markers+text", name="Actual Admissions",
    text=[f"{v:,.0f}" for v in monthly_comparison["Actual Admissions"]],
    textposition="top center"
))
fig1.add_trace(go.Scatter(
    x=monthly_comparison["Month"], y=monthly_comparison["Predicted Admissions"],
    mode="lines+markers+text", name="Predicted Admissions",
    text=[f"{v:,.0f}" for v in monthly_comparison["Predicted Admissions"]],
    textposition="bottom center"
))
fig1.update_layout(
    title="Actual vs Predicted Monthly Admissions — 2026 (Synthetic Data)",
    xaxis_title="Month", yaxis_title="Admissions",
    template="plotly_white", height=550, hovermode="x unified"
)
fig1.show()

# 9. Graph 2 — Channel-wise Actual vs Predicted (2026)
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=channel_comparison["Channel"], y=channel_comparison["Actual Admissions"],
    name="Actual Admissions",
    text=[f"{v:,.0f}" for v in channel_comparison["Actual Admissions"]],
    textposition="outside"
))
fig2.add_trace(go.Bar(
    x=channel_comparison["Channel"], y=channel_comparison["Predicted Admissions"],
    name="Predicted Admissions",
    text=[f"{v:,.0f}" for v in channel_comparison["Predicted Admissions"]],
    textposition="outside"
))
fig2.update_layout(
    title="Channel-wise Actual vs Predicted Admissions — 2026 (Synthetic Data)",
    xaxis_title="Channel", yaxis_title="Admissions",
    barmode="group", template="plotly_white", height=550
)
fig2.show()

# 10. Export an illustrative synthetic report
output_path = Path("outputs/synthetic_admission_forecast_report.xlsx")
output_path.parent.mkdir(exist_ok=True)
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    monthly_comparison.to_excel(writer, sheet_name="Monthly Comparison", index=False)
    channel_comparison.to_excel(writer, sheet_name="Channel Comparison", index=False)
    overall_stats.to_excel(writer, sheet_name="Overall Performance", index=False)
    channel_stats.to_excel(writer, sheet_name="Channel Performance", index=False)
    best_channel_models.to_excel(writer, sheet_name="Selected Models", index=False)
    future_predictions.to_excel(writer, sheet_name="Nov-Dec Forecast")
print("Synthetic report saved to:", output_path)
print("All data, metrics, forecasts, and charts are fictional demonstration outputs.")

Saving synthetic_admission_data.xlsx to synthetic_admission_data.xlsx
OVERALL MODEL PERFORMANCE — SYNTHETIC DATA


,Model,MAE,RMSE,MAPE (%),WAPE (%),Bias,Bias (%)
0,ETS,55.51,75.37,1.85,1.92,-14.99,-0.52
1,Recent Average,219.63,321.61,7.55,7.58,28.63,0.99


CHANNEL-WISE MODEL PERFORMANCE — SYNTHETIC DATA


,Channel,Model,MAE,RMSE,MAPE (%),WAPE (%),Bias,Bias (%)
0,CGHS,ETS,98.62,119.65,2.30,2.33,-38.52,-0.91
1,CGHS,Recent Average,394.68,489.90,9.64,9.31,56.62,1.34
2,Cash,ETS,57.10,65.32,1.51,1.53,-11.59,-0.31
3,Cash,Recent Average,333.95,388.34,9.22,8.95,23.90,0.64
4,Corporate,ETS,55.59,65.72,1.88,1.85,18.38,0.61
5,Corporate,Recent Average,227.10,297.00,7.83,7.57,56.30,1.88
6,EWS,ETS,11.69,15.40,1.35,1.36,-0.62,-0.07
7,EWS,Recent Average,65.72,79.57,7.79,7.65,13.78,1.60
8,International,ETS,12.63,16.89,2.11,2.11,-3.41,-0.57
9,International,Recent Average,32.85,43.13,5.54,5.50,1.20,0.20


BEST MODEL BY CHANNEL — SYNTHETIC DATA


,Channel,Model,WAPE (%),MAPE (%)
0,EWS,ETS,1.36,1.35
1,Cash,ETS,1.53,1.51
2,Corporate,ETS,1.85,1.88
3,TPA,ETS,1.97,1.95
4,International,ETS,2.11,2.11
5,CGHS,ETS,2.33,2.30


... (3 outputs left out)

,Month,Actual Admissions,Predicted Admissions
0,Jan 2026,18970,18765.0
1,Feb 2026,17439,17452.0
2,Mar 2026,19090,18774.0
3,Apr 2026,18279,18060.0
4,May 2026,18388,18583.0
5,Jun 2026,18033,17492.0
6,Jul 2026,17338,17327.0
7,Aug 2026,15853,16145.0
8,Sep 2026,14998,14788.0
9,Oct 2026,15442,15544.0


CHANNEL-WISE ACTUAL VS PREDICTED — SYNTHETIC DATA


,Channel,Actual Admissions,Predicted Admissions
0,Cash,37306.0,37190.0
1,CGHS,42406.0,42021.0
2,Corporate,30006.0,30190.0
3,EWS,8597.0,8591.0
4,International,5972.0,5938.0
5,TPA,49543.0,49001.0


Synthetic report saved to: outputs/synthetic_admission_forecast_report.xlsx
All data, metrics, forecasts, and charts are fictional demonstration outputs.
